# Task 1 — WiFo2 architecture explorer

What this notebook does: builds the baseline model, traces a real CSI sample
through every stage, and shows exactly which parts do the LoS/NLoS
classification.

Backbone is **WiFo2** — a masked-autoencoder ViT (`model.py:597`). Task 1 uses
the **encoder only** plus a single linear head. The decoder, ResNet-18 and
FastDepthV2 belong to Tasks 2 and 3 and are never touched here.

```
(B,2,24,8,128) -> patchify -> Linear embed -> +SinCos_3D pos -> 6x Block
               -> LayerNorm -> flatten -> Linear(24576,2) -> (B,2)
```

Kernel: `itu-challenge`. Run from the repo root.

In [ ]:
import sys, warnings, json
from pathlib import Path
from types import SimpleNamespace

warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, torch, torch.nn as nn, h5py
import matplotlib.pyplot as plt

from model import WiFo2_model
from main import freeze_wifo

torch.manual_seed(0)
device = torch.device('cpu')          # 'mps' works too; cpu keeps numbers reproducible
print('torch', torch.__version__, '| root', ROOT)

## 1. Build the model

`WiFo2_model` picks a size preset (`model.py:24`). `main.py` defaults to
`tinypro`, so that is the baseline. Everything below mirrors the defaults in
`create_argparser()`.

In [ ]:
args = SimpleNamespace(
    size='tinypro',            # main.py:53
    patch_size=4,              # spatial patch (antenna x subcarrier)
    t_patch_size=4,            # temporal patch (time slots)
    no_qkv_bias=0,
    pos_emb='SinCos_3D',
    MoE=False,
    csi_enhance=1,
    rgb_pos_enhance=1,
    fastdepth_weights=str(ROOT / 'weights/FastDepthV2_L1_Best.pth'),
    device_id='0',
)

model = WiFo2_model(args=args).to(device).eval()

### Size presets available

Fourteen are defined; only `tinypro` is wired as the default. Swapping `size`
above is the cheapest architectural experiment in this repo.

In [ ]:
import re
src = (ROOT / 'model.py').read_text()
rows = []
for name, body in re.findall(r"(?:if|elif) args\.size == '([\w\-.]+)':\s*\n\s*model = WiFo2\((.*?)\n\s*\)", src, re.S):
    kv = dict(re.findall(r"(embed_dim|depth|decoder_embed_dim|decoder_depth|num_heads|mlp_ratio)=(\d+)", body))
    rows.append((name, kv))

hdr = ['embed_dim', 'depth', 'decoder_embed_dim', 'decoder_depth', 'num_heads', 'mlp_ratio']
print(f"{'size':18s}" + ''.join(f'{h:>19s}' for h in hdr))
for name, kv in rows:
    mark = ' <- default' if name == args.size else ''
    print(f'{name:18s}' + ''.join(f"{kv.get(h,'-'):>19s}" for h in hdr) + mark)

## 2. Load the pretrained checkpoint

`weights/model_best.pkl` holds **only the WiFo2 backbone** — 183 tensors,
~324k params. It is loaded with `strict=False` (`main.py:94`) because the task
heads, ResNet-18 and FastDepthV2 are not in it:

- heads (`Fine_Tune_Layer_*`, `pos_align_layer`) start **randomly initialised** — that is what you fine-tune
- `res18` comes from torchvision ImageNet weights, `fast_depth` from its own `.pth`

So a long `missing_keys` list here is expected, not a bug. What matters is
`unexpected_keys == 0` and that no `blocks.*` key is missing.

In [ ]:
sd = torch.load(ROOT / 'weights/model_best.pkl', map_location='cpu')
msg = model.load_state_dict(sd, strict=False)

print(f'checkpoint : {len(sd)} tensors, {sum(v.numel() for v in sd.values()):,} params')
print(f'missing    : {len(msg.missing_keys)}   unexpected: {len(msg.unexpected_keys)}')

import collections
own = collections.Counter(k.split('.')[0] for k in msg.missing_keys)
print('\nmissing keys grouped by top-level module:')
for k, v in own.most_common():
    print(f'  {k:28s} {v}')

backbone_missing = [k for k in msg.missing_keys if k.startswith(('blocks.', 'Embedding_encoder.', 'norm.'))]
print('\nencoder keys missing from checkpoint:', backbone_missing or 'none  <- backbone loaded cleanly')

## 3. Parameter budget

15.7M params total, but Task 1 trains **49,154** of them — one `Linear` layer.
`freeze_wifo` (`main.py:120`) freezes everything, then unfreezes only
`Fine_Tune_Layer_LoS_NLoS`.

In [ ]:
freeze_wifo(args, model, task_id=1)

groups = {
    'encoder blocks (6x)': model.blocks,
    'patch embed'        : model.Embedding_encoder,
    'encoder norm'       : model.norm,
    'pos embeds'         : [model.pos_embed_spatial, model.pos_embed_temporal],
    'TASK 1 head'        : model.Fine_Tune_Layer_LoS_NLoS,
    'decoder (task 2)'   : model.decoder_blocks,
    'resnet18 (task 2)'  : model.res18,
    'fastdepth (task 3)' : model.fast_depth,
}

def count(m):
    ps = m if isinstance(m, list) else list(m.parameters())
    return sum(p.numel() for p in ps), sum(p.numel() for p in ps if p.requires_grad)

print(f"{'module':22s}{'params':>14s}{'trainable':>12s}{'used by task 1':>17s}")
used = {'encoder blocks (6x)', 'patch embed', 'encoder norm', 'pos embeds', 'TASK 1 head'}
for name, m in groups.items():
    tot, tr = count(m)
    print(f'{name:22s}{tot:>14,}{tr:>12,}{("yes" if name in used else "no"):>17s}')

tot = sum(p.numel() for p in model.parameters())
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n{"TOTAL":22s}{tot:>14,}{tr:>12,}')
print(f'trainable fraction: {100*tr/tot:.3f}%')

## 4. Load real data

`H` is complex `(10, 24, 8, 128)`. `LoadBatch` in `DataLoader.py` splits
real/imag into 2 channels and rearranges to `b o t n k`, giving the model input
`(B, 2, 24, 8, 128)`.

In [ ]:
def load_mat(path, key):
    with h5py.File(path, 'r') as f:
        a = f[key][...]
    if a.dtype.names == ('real', 'imag'):
        a = a['real'] + 1j * a['imag']
    return a.transpose(range(a.ndim)[::-1])

H = load_mat(ROOT / 'dataset/Task1/H_train.mat', 'H')       # (10,24,8,128) complex
L = load_mat(ROOT / 'dataset/Task1/L_train.mat', 'label').ravel().astype(int)

x = torch.stack([torch.tensor(H.real), torch.tensor(H.imag)], dim=1).float().to(device)
y = torch.tensor(L).long()

print('H     ', H.shape, H.dtype)
print('labels', L, ' (2 classes: LoS vs NLoS; which integer is which is not stated in this repo)')
print('model input x', tuple(x.shape), '  = (B, real|imag, time, antenna, subcarrier)')

## 5. Patchify — how 24,576 numbers become 384 tokens

`patch_size=4`, `t_patch_size=4`, so one token covers a
**4 time slots x 4 antennas x 4 subcarriers** cube, both real and imag.

```
time       24 / 4 = 6
antenna     8 / 4 = 2
subcarrier 128 / 4 = 32
                     -------
tokens      6 x 2 x 32 = 384       each 2 x 4x4x4 = 128 numbers
```

384 x 128 = 49,152 = 2 x 24,576. Nothing is discarded — patchify is a pure
reshape.

In [ ]:
patches = model.patchify(x)
print('patchify   ', tuple(patches.shape), ' = (B, tokens, real|imag, voxels-per-patch)')

B, Ntok, C2, V = patches.shape
print(f'\ntokens      {Ntok}  = {24//4} time x {8//4} antenna x {128//4} subcarrier')
print(f'per token   {C2} x {V} = {C2*V} numbers')
print(f'total       {Ntok*C2*V:,}  vs raw sample {2*24*8*128:,}  -> lossless reshape')

flat = patches.reshape(B, Ntok, -1)
emb = model.Embedding_encoder(flat)
print(f'\nEmbedding_encoder: Linear({model.Embedding_encoder.in_features} -> {model.Embedding_encoder.out_features})')
print('after embed', tuple(emb.shape))

## 6. Masking — why Task 1 passes `mask_ratio=0.0`

The encoder always routes through a masking function (`forward_encoder`,
`model.py:1129`). Task 1 calls it with `mask_strategy='fre'` and
`mask_ratio=0.0` (`flops.py:36`), so **nothing is dropped** — all 384 tokens
survive and the backbone acts as a plain feature extractor.

Masking matters during MAE *pretraining*, where the model learns to reconstruct
hidden patches. For classification it is switched off.

In [ ]:
from mask_strategy import fre_masking

for ratio in [0.0, 0.25, 0.5]:
    kept, mask, ids_restore, ids_keep = fre_masking(emb.clone(), ratio, T=6, H=2, W=32)
    print(f'mask_ratio={ratio:<5} kept tokens {kept.shape[1]:>4} / {Ntok}   masked {int(mask.sum().item()/B):>4}')

print('\nTask 1 uses mask_ratio=0.0 -> full 384 tokens reach the head.')

## 7. Full shape trace through the encoder

Hooks on every stage. This is the whole Task 1 forward path.

In [ ]:
trace = []
hooks = []

def watch(name, module):
    def fn(_m, _i, o):
        t = o[0] if isinstance(o, tuple) else o
        trace.append((name, tuple(t.shape)))
    hooks.append(module.register_forward_hook(fn))

watch('Embedding_encoder', model.Embedding_encoder)
for i, blk in enumerate(model.blocks):
    watch(f'  block {i} (attn+mlp)', blk)
watch('encoder LayerNorm', model.norm)
watch('Fine_Tune_Layer_LoS_NLoS', model.Fine_Tune_Layer_LoS_NLoS)

with torch.inference_mode():
    logits = model(x, mask_ratio=0.0, mask_strategy='fre', task_id=1)

for h in hooks:
    h.remove()

print(f"{'stage':30s}{'output shape':>22s}")
print(f"{'input (B,2,T,A,K)':30s}{str(tuple(x.shape)):>22s}")
print(f"{'patchify':30s}{str(tuple(patches.shape)):>22s}")
for name, shp in trace:
    print(f'{name:30s}{str(shp):>22s}')
print(f'\nlogits {tuple(logits.shape)} -> softmax -> P(class)')

## 8. Anatomy of one encoder block

Standard pre-norm ViT block (`Block`, `model.py:450`) — LayerNorm, multi-head
self-attention, residual, LayerNorm, MLP, residual. `tinypro` uses
`mlp_ratio=1`, so the MLP hidden width equals `embed_dim` (64), unusually
narrow.

Self-attention runs over all 384 tokens at once, so a single block can relate
any (time, antenna, subcarrier) patch to any other. That is where cross-axis
structure — e.g. "this delay spread pattern across frequency, steady over
time" — gets picked up.

In [ ]:
blk = model.blocks[0]
print(blk)
print(f'\nheads {model.num_heads}  head_dim {model.embed_dim // model.num_heads}  mlp_ratio {model.mlp_ratio}')
print(f'params per block {sum(p.numel() for p in blk.parameters()):,}  x{len(model.blocks)} blocks')

## 9. The Task 1 head

One linear layer. It flattens **all 384 tokens x 64 dims = 24,576 features**
into 2 logits — no pooling, no CLS token.

```python
self.Fine_Tune_Layer_LoS_NLoS = nn.Sequential(nn.Linear(384*embed_dim, 2))
```

Note the hardcoded `384`: change `patch_size`, `t_patch_size`, or the input
dimensions and this layer breaks. It is tied to `(24, 8, 128)` input with 4x4x4
patches.

In [ ]:
head = model.Fine_Tune_Layer_LoS_NLoS[0]
print(f'Linear({head.in_features} -> {head.out_features})  params {sum(p.numel() for p in head.parameters()):,}')
print(f'in_features = 384 tokens x {model.embed_dim} embed_dim = {384*model.embed_dim}')

with torch.inference_mode():
    latent, mask, ids_restore, input_size, _ = model.forward_encoder(x, 0.0, 'fre', scale=[1,1,1])

print('\nlatent     ', tuple(latent.shape), ' (B, tokens, embed_dim)')
print('input_size ', input_size, ' = (T/4, A/4, K/4) patch grid')
print('flatten(1) ', tuple(latent.flatten(1).shape), '-> head')

## 10. Predictions before fine-tuning

The head is randomly initialised, so these logits are noise. Shown to confirm
the plumbing works end to end and to give a baseline for after training.

In [ ]:
with torch.inference_mode():
    logits = model(x, mask_ratio=0.0, mask_strategy='fre', task_id=1)
    prob = logits.softmax(-1)

pred = prob.argmax(-1)
print(f"{'sample':>7}{'true':>7}{'pred':>7}{'P(0)':>9}{'P(1)':>9}")
for i in range(len(y)):
    print(f'{i:>7}{y[i].item():>7}{pred[i].item():>7}{prob[i,0]:>9.3f}{prob[i,1]:>9.3f}')

print(f'\naccuracy {(pred == y).float().mean():.2f}  (random head - expected ~chance)')

## 11. Are the frozen features already separable?

The interesting question: does the pretrained backbone, with no Task 1
training at all, produce embeddings where LoS and NLoS already fall apart? If
yes, the single linear head is enough. If no, the head cannot fix it and you
need to unfreeze more.

Only 10 samples here, so read this as a smoke test, not evidence.

In [ ]:
feat = latent.flatten(1).numpy()                     # (10, 24576) frozen features
feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-8)

# 2D view via SVD
u, s, vt = np.linalg.svd(feat - feat.mean(0), full_matrices=False)
proj = u[:, :2] * s[:2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for cls, marker in [(0, 'o'), (1, '^')]:
    m = L == cls
    ax1.scatter(proj[m, 0], proj[m, 1], marker=marker, s=90, label=f'class {cls}')
for i in range(len(L)):
    ax1.annotate(str(i), proj[i], textcoords='offset points', xytext=(6, 4), fontsize=8)
ax1.set(title='frozen WiFo2 features, top 2 SVD components', xlabel='PC1', ylabel='PC2')
ax1.legend(); ax1.grid(alpha=.3)

# cosine similarity between samples
fn = feat / np.linalg.norm(feat, axis=1, keepdims=True)
im = ax2.imshow(fn @ fn.T, cmap='RdBu_r', vmin=-1, vmax=1)
ax2.set(title='cosine similarity (samples ordered by index)', xticks=range(10), yticks=range(10))
ax2.set_xticklabels(L); ax2.set_yticklabels(L)
ax2.set_xlabel('label'); ax2.set_ylabel('label')
plt.colorbar(im, ax=ax2)
plt.tight_layout(); plt.show()

same = [(fn[i] @ fn[j]) for i in range(10) for j in range(i+1, 10) if L[i] == L[j]]
diff = [(fn[i] @ fn[j]) for i in range(10) for j in range(i+1, 10) if L[i] != L[j]]
print(f'mean cosine, same label : {np.mean(same):+.4f}')
print(f'mean cosine, diff label : {np.mean(diff):+.4f}')
print(f'gap                     : {np.mean(same) - np.mean(diff):+.4f}  (positive = backbone separates classes)')

## 12. Fine-tuning recipe

What `main.py` actually does for Task 1:

| step | code |
|---|---|
| build | `WiFo2_model(args)` with `size='tinypro'` |
| load | `load_state_dict(model_best.pkl, strict=False)` |
| freeze | `freeze_wifo(args, model, task_id=1)` — only the head trains |
| optimise | `Adam(trainable, lr=1e-5)`, `weight_decay=0`, `clip_grad=0.05` |
| loop | `TrainLoop(...).run()` in `train.py`, `epochs=200`, `early_stop=5` |

Run it:

```bash
python main.py --task_id 1 --task Train --size tinypro --batch_size 64
```

The cell below runs the same optimisation by hand on the 10 training samples,
purely to show the mechanics. Ten samples cannot generalise — expect
memorisation.

In [ ]:
head = model.Fine_Tune_Layer_LoS_NLoS
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
lossf = nn.CrossEntropyLoss()

# backbone is frozen, so cache its output once instead of recomputing each step
with torch.inference_mode():
    feats = model.forward_encoder(x, 0.0, 'fre', scale=[1,1,1])[0].flatten(1)
feats = feats.clone()

for ep in range(101):
    opt.zero_grad()
    out = head(feats)
    loss = lossf(out, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 0.05)
    opt.step()
    if ep % 20 == 0:
        acc = (out.argmax(-1) == y).float().mean()
        print(f'epoch {ep:>3}  loss {loss.item():.4f}  acc {acc:.2f}')

print('\nOverfitting 10 samples. Real training needs the full challenge dataset.')

## Summary

| question | answer |
|---|---|
| backbone | WiFo2, MAE-style ViT, `size=tinypro` |
| depth / width | 6 blocks, `embed_dim=64`, 4 heads, `mlp_ratio=1` |
| tokens | 384, each a 4x4x4 (time x antenna x subcarrier) patch of real+imag |
| pretrained from | `weights/model_best.pkl`, backbone only (~324k params) |
| Task 1 head | `Linear(24576, 2)`, 49,154 params, randomly initialised |
| decoder used? | no — encoder only |
| masking | `fre` strategy at `mask_ratio=0.0`, so effectively off |
| trainable | 49,154 of 15,713,661 (0.31%) |

Places worth experimenting:

- **`size`** — `littlepro`/`smallpro` widen the backbone, but `model_best.pkl` only matches `tinypro` dims, so a bigger preset means training from scratch
- **`freeze_wifo`** — unfreezing the last encoder block often beats a linear probe alone
- **head** — mean-pool over tokens instead of flattening 24,576 features; far fewer params, and it survives a change of input size
- **`mask_ratio`** — non-zero acts as token dropout, a cheap regulariser